# Chapter 1: Advanced Document Ingestion and Hybrid Search (RRF)

In standard RAG pipelines, simple top-k retrieval often struggles with multi-topic documents. This notebook implements:
1. **Parent-Child Chunking**: Break large documents into 1000-char parent blocks, and subdivided overlapping 250-char child blocks. We search the child blocks to get fine-grained vector similarity, but feed the parent block to the LLM to preserve holistic context.
2. **Dense Vector Search**: Using SentenceTransformers cosine similarity.
3. **Sparse BM25 Search**: Using keyword lexical match.
4. **Reciprocal Rank Fusion (RRF)**: A mathematically robust blending function to merge dense and sparse rankings.

$$\text{RRF Score}(d) = \sum_{m \in \text{Retrievers}} \frac{1}{k_{rrf} + r_m(d)}$$
where $r_m(d)$ is the rank of document $d$ in system $m$, and $k_{rrf} \approx 60$ is a constant.

In [ ]:
import os
import sys
# Append parent path to allow direct src imports
sys.path.append(os.path.dirname(os.getcwd()))

from src.ingestion.chunking import load_and_chunk_corpus
from src.retrieval.hybrid import HybridRetriever

corpus_path = os.path.join(os.path.dirname(os.getcwd()), "data", "sample_corpus")
print(f"Target corpus directory: {corpus_path}")

### Step 1: Parent-Child Chunking
Let's load the raw corpus files and chunk them. We will see the ratio of parents to child nodes.

In [ ]:
corpus_data = load_and_chunk_corpus(corpus_path)
parents = corpus_data["parents"]
children = corpus_data["children"]

print(f"\nCreated {len(parents)} Parent Chunks (1000 characters)")
print(f"Created {len(children)} Child Chunks (250 characters)")

# Let's print a sample parent-child mapping
if children:
    sample_child = children[0]
    pid = sample_child["metadata"]["parent_id"]
    print(f"\nSample Child Text: '{sample_child['text']}'")
    print(f"Linked to Parent ID: {pid}")

### Step 2: Indexing & Hybrid Search with RRF
We instantiate the `HybridRetriever`, build vector & BM25 indices, and execute hybrid RRF search mapped back to parent context.

In [ ]:
retriever = HybridRetriever()
retriever.index_corpus(corpus_data)

# Perform hybrid retrieval
query = "What are the rules regarding hybrid remote work stipends?"
results = retriever.retrieve(query, k=3, top_n=2)

print(f"\nTop Hybrid Search Results (Translated back to Parent context):")
for rank, doc in enumerate(results):
    print(f"\nRank {rank+1} (RRF Score: {doc['score']:.5f})")
    print(f"Source: {doc['metadata']['source']}")
    print(f"Content: {doc['text'][:300]}...")